In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [2]:
alimentos = ['A','B']

restrições = {
    'x':2,
    'y':64,
    'z':34
}

informações={
    'A':{
        'x':10,
        'y':40,
        'z':50,
        'custo':0.60,
    },
    'B':{
        'x':20,
        'y':60,
        'z':20,
        'custo':0.80,
    }
}

In [9]:
model = pyo.ConcreteModel()

model.alimentos = pyo.Set(initialize=alimentos)
model.nutrientes = pyo.Set(initialize=restrições.keys())

model.x = pyo.Var(model.alimentos, bounds=(0,None), domain=NonNegativeReals)

def objetivo(model):
    return sum(
        model.x[a] * informações[a]['custo'] for a in model.alimentos
    )
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.minimize)

def restrição(model, n):
    return sum(
        model.x[a] * informações[a][n] for a in model.alimentos
    ) >= restrições[n]
model.restrição = pyo.Constraint(model.nutrientes, rule=restrição)

In [10]:
model.pprint()

2 Set Declarations
    alimentos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'A', 'B'}
    nutrientes : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'x', 'y', 'z'}

1 Var Declarations
    x : Size=2, Index=alimentos
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          A :     0 :  None :  None : False :  True : NonNegativeReals
          B :     0 :  None :  None : False :  True : NonNegativeReals

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 0.6*x[A] + 0.8*x[B]

1 Constraint Declarations
    restrição : Size=3, Index=nutrientes, Active=True
        Key : Lower : Body              : Upper : Active
          x :   2.0 : 10*x[A] + 20*x[B] :  +Inf :   True
          y :  64.0 : 40*x[A] + 60*x[B] :  +Inf :

In [11]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp55l67ueq.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp8fgtk_st.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp8fgtk_st.pyomo.lp
Objective sense      : Minimize
Variables            :       2
Objective nonzeros   :       2
Linear constraints   :       3  [Greater: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 0.6000000        Max   : 0.8000000

In [20]:
for x in model.x:
    print(f'{x} = {model.x[x].value *informações[x]["custo"]:.2f}')

A = 0.21
B = 0.67


In [15]:
model.objetivo()

0.8763636363636363